In [19]:
import requests
from urllib.parse import unquote

url = 'https://apis.data.go.kr/6260000/BusanCommercialCountingService/getCommercialCountingList'

service_key = 'mgnDSX5DP1DCeTiy5HhxKwhy0IkUeZDFUw%2Bib%2FftCc7dttQTy1aSMFcm8dtO%2BekyERqXPh84QRrxcL9OvFqDXg%3D%3D'

# 이미 Encoding된 키를 한 번 Decoding
service_key = unquote(service_key)

params = {
    'serviceKey': service_key,
    'pageNo': 1,
    'numOfRows': 10,
    'resultType': 'json'
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.text)

200
{"response":{"header":{"resultMsg":"NORMAL_CODE","resultCode":"00"},"body":{"totalCount":"384","items":{"item":[{"type_nm":"발달상권","comm_nm":"부산 부산진구 양정역_2","fac_total":"47","gov_total":"2","sch_total":"0","hos_total":"25","bus_total":"13","sub_total":"0","hot_total":"6","sho_total":"1"},{"type_nm":"발달상권","comm_nm":"부산 부산진구 전포역","fac_total":"15","gov_total":"0","sch_total":"0","hos_total":"2","bus_total":"11","sub_total":"0","hot_total":"1","sho_total":"1"},{"type_nm":"발달상권","comm_nm":"부산 북구 덕천역_1","fac_total":"36","gov_total":"0","sch_total":"0","hos_total":"14","bus_total":"16","sub_total":"0","hot_total":"6","sho_total":"0"},{"type_nm":"발달상권","comm_nm":"부산 북구 덕천역_2","fac_total":"90","gov_total":"1","sch_total":"0","hos_total":"72","bus_total":"16","sub_total":"0","hot_total":"1","sho_total":"0"},{"type_nm":"발달상권","comm_nm":"부산 북구 덕천역_3","fac_total":"16","gov_total":"0","sch_total":"0","hos_total":"4","bus_total":"7","sub_total":"0","hot_total":"5","sho_total":"0"},{"type_nm":"발달상

In [20]:
import pandas as pd

data = response.json()

items = data['response']['body']['items']['item']

df = pd.DataFrame(items)

df.head()

,type_nm,comm_nm,fac_total,gov_total,sch_total,hos_total,bus_total,sub_total,hot_total,sho_total
0,발달상권,부산 부산진구 양정역_2,47,2,0,25,13,0,6,1
1,발달상권,부산 부산진구 전포역,15,0,0,2,11,0,1,1
2,발달상권,부산 북구 덕천역_1,36,0,0,14,16,0,6,0
3,발달상권,부산 북구 덕천역_2,90,1,0,72,16,0,1,0
4,발달상권,부산 북구 덕천역_3,16,0,0,4,7,0,5,0


In [21]:
df.to_csv(
    '/content/busan_facility.csv',
    index=False,
    encoding='utf-8-sig'
)

In [22]:
from google.colab import files

files.download('/content/busan_facility.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
import requests
import pandas as pd

all_items = []

page = 1
num_of_rows = 100

while True:

    params = {
        'serviceKey': service_key,
        'pageNo': page,
        'numOfRows': num_of_rows,
        'resultType': 'json'
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print('오류:', response.status_code)
        print(response.text)
        break

    data = response.json()

    body = data['response']['body']

    items = body['items']['item']

    if isinstance(items, dict):
        items = [items]

    all_items.extend(items)

    total_count = int(body['totalCount'])

    print(f'{page}페이지 수집: {len(items)}개 / 누적 {len(all_items)}개')

    if len(all_items) >= total_count:
        break

    page += 1

df = pd.DataFrame(all_items)

print('최종 데이터:', df.shape)

1페이지 수집: 100개 / 누적 100개
2페이지 수집: 100개 / 누적 200개
3페이지 수집: 100개 / 누적 300개
4페이지 수집: 84개 / 누적 384개
최종 데이터: (384, 10)


In [34]:
df = df.rename(columns={
    'type_nm': '상권구분',
    'comm_nm': '상권명',
    'fac_total': '집객시설수',
    'gov_total': '관공서수',
    'sch_total': '학교시설수',
    'hos_total': '병원시설수',
    'bus_total': '교통시설수',
    'sub_total': '지하철수',
    'hot_total': '숙박시설수',
    'sho_total': '쇼핑시설수'
})

print(df.columns)

Index(['상권구분', '상권명', '집객시설수', '관공서수', '학교시설수', '병원시설수', '교통시설수', '지하철수',
       '숙박시설수', '쇼핑시설수'],
      dtype='object')


In [35]:
df.to_csv(
    '/content/busan_facility.csv',
    index=False,
    encoding='utf-8-sig'
)

print('CSV 저장 완료')

CSV 저장 완료


In [36]:
from google.colab import files

files.download('/content/busan_facility.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>